In [ ]:
from pathlib import Path
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from tqdm import tqdm

# ============================================================
# PATH CONFIGURATION  (portable — no hard-coded local paths)
# ============================================================
REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "citysegmentdeprivation" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DATA_EXTERNAL              = REPO_ROOT / "data_external"
ZENODO_DATA                = DATA_EXTERNAL / "zenodo"
UCDB_DIR                   = DATA_EXTERNAL / "ucdb"
GHSPOP_DIR                 = DATA_EXTERNAL / "ghspop"

OUTPUT_TABLES              = REPO_ROOT / "outputs" / "tables" / "revision2"
OUTPUT_TABLES_INTERMEDIATE = OUTPUT_TABLES / "intermediate"
OUTPUT_FIGURES             = REPO_ROOT / "outputs" / "figures" / "revision2"

OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_INTERMEDIATE.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

# ============================================================
# USER INPUTS
# ============================================================
UCDB_GPKG  = UCDB_DIR / "GHS_STAT_UCDB2015MT_GLOBE_R2019A_V1_2.gpkg"   # UCDB 2019 GPKG
UCDB_LAYER = None               # Set layer name if known, else auto-detected
POP_TIF    = GHSPOP_DIR / "GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0.tif"  # GHS-POP 2025 raster
ID_COL     = "ID_HDC_G0"
OUT_COL    = "GHSPOP2023"

# Output: derived GPKG is large — stored under data_external/zenodo/, NOT committed to GitHub.
# Archive this file on Zenodo; see notebooks/revision2_coverage/README.md for details.
OUT_GPKG  = ZENODO_DATA / "GHS_STAT_UCDB2015MT_GLOBE_R2019A_V1_2_with_GHSPOP2023.gpkg"
OUT_LAYER = "ucdb2019_with_pop2023"

In [2]:
# Detect layer if not specified
layers = gpd.list_layers(UCDB_GPKG)
if UCDB_LAYER is None:
    UCDB_LAYER = layers.iloc[0]["name"]

print("Using UCDB layer:", UCDB_LAYER)

ucdb = gpd.read_file(UCDB_GPKG, layer=UCDB_LAYER)
print("UCDB polygons:", len(ucdb))

with rasterio.open(POP_TIF) as src:
    raster_crs = src.crs
    raster_res = src.res
    raster_nodata = src.nodata

print("Raster CRS:", raster_crs)
print("Raster resolution:", raster_res)
print("Raster nodata:", raster_nodata)

# Reproject polygons to raster CRS
ucdb_r = ucdb.to_crs(raster_crs).copy()
ucdb_r = ucdb_r.reset_index(drop=True)
ucdb_r[ID_COL] = ucdb_r[ID_COL].astype(str)

Using UCDB layer: GHS_STAT_UCDB2015MT_GLOBE_R2019A_V1_2
UCDB polygons: 13135
Raster CRS: ESRI:54009
Raster resolution: (100.0, 100.0)
Raster nodata: -200.0


In [3]:
# Create spatial index
sidx = ucdb_r.sindex

# Assign integer label per polygon (1..N)
N = len(ucdb_r)
labels = np.arange(1, N + 1, dtype=np.int32)

# Result accumulator
pop_sum = np.zeros(N + 1, dtype=np.float64)  # index 0 = background

In [4]:
with rasterio.open(POP_TIF) as src:
    
    total_windows = sum(1 for _ in src.block_windows(1))
    print("Total raster blocks:", total_windows)
    
    for _, window in tqdm(src.block_windows(1), total=total_windows):
        
        # Window bounds in map coordinates
        bounds = rasterio.windows.bounds(window, src.transform)
        
        # Find polygons intersecting this block
        candidate_idx = list(sidx.intersection(bounds))
        if not candidate_idx:
            continue
        
        # Read population values for window
        pop = src.read(1, window=window, masked=False).astype(np.float64)
        
        if raster_nodata is not None:
            pop[pop == raster_nodata] = 0.0
        
        # Transform for this window
        win_transform = rasterio.windows.transform(window, src.transform)
        
        # Prepare shapes for rasterization
        shapes = [
            (ucdb_r.geometry.iloc[i], int(labels[i]))
            for i in candidate_idx
            if not ucdb_r.geometry.iloc[i].is_empty
        ]
        
        if not shapes:
            continue
        
        # Rasterize polygon labels into this window
        label_raster = rasterize(
            shapes=shapes,
            out_shape=(window.height, window.width),
            transform=win_transform,
            fill=0,
            dtype=np.int32,
            all_touched=False  # fastest + consistent with grid alignment
        )
        
        mask = label_raster > 0
        if not np.any(mask):
            continue
        
        # Accumulate population sums by polygon label
        bc = np.bincount(
            label_raster[mask].ravel(),
            weights=pop[mask].ravel(),
            minlength=N + 1
        )
        
        pop_sum += bc

print("Finished population aggregation.")

Total raster blocks: 992640


100%|██████████| 992640/992640 [00:27<00:00, 36572.86it/s] 


Finished population aggregation.


In [5]:
# Drop background index 0
ucdb[OUT_COL] = pop_sum[1:]

print(ucdb[[OUT_COL]].describe())

         GHSPOP2023
count  1.313500e+04
mean   2.718931e+05
std    1.228593e+06
min    0.000000e+00
25%    4.931210e+04
50%    8.410470e+04
75%    1.649525e+05
max    4.209125e+07


In [6]:
ucdb.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")
print("Saved:", OUT_GPKG)
print("Layer:", OUT_LAYER)

Saved: D:\VSG\DIRTY_MODEL\February2026\GHSUCDB_Analysis\GHS_STAT_UCDB2015MT_GLOBE_R2019A\GHS_STAT_UCDB2015MT_GLOBE_R2019A_V1_2_with_GHSPOP2023.gpkg
Layer: ucdb2019_with_pop2023
